In [1]:
!pip install scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt
import os

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, learning_curve
)
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("Imports successful")

Imports successful


In [2]:
data         = joblib.load("/kaggle/input/datasets/pes2ug23cs197/rf-splits/splits.pkl")
X_train      = data["X_train"]
X_val        = data["X_val"]
X_test       = data["X_test"]
y_train      = data["y_train"]
y_val        = data["y_val"]
y_test       = data["y_test"]
del data
gc.collect()

le           = joblib.load("/kaggle/input/datasets/pes2ug23cs197/rf-splits/label_encoder.pkl")
feature_cols = joblib.load("/kaggle/input/datasets/pes2ug23cs197/rf-splits/feature_cols.pkl")
num_classes  = len(le.classes_)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Classes: {num_classes} | Features: {X_train.shape[1]}")

Train: 34999 | Val: 7500 | Test: 7500
Classes: 1117 | Features: 1756


In [3]:
rf_model = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 50,
    min_samples_leaf = 2,
    max_features     = "sqrt",
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)

joblib.dump(rf_model, "rf_model.pkl")
print("rf_model.pkl saved")

Training Random Forest...
rf_model.pkl saved


In [4]:
y_val_pred  = rf_model.predict(X_val)
val_acc     = accuracy_score(y_val, y_val_pred)
train_acc   = accuracy_score(y_train, rf_model.predict(X_train))
gap         = train_acc - val_acc

print(f"Train Accuracy : {train_acc*100:.2f}%")
print(f"Val   Accuracy : {val_acc*100:.2f}%")
print(f"Gap            : {gap*100:.2f}%  {'- OK' if gap < 0.10 else '- Check regularization'}")

Train Accuracy : 90.65%
Val   Accuracy : 82.56%
Gap            : 8.09%  - OK


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_model, X_train, y_train,
    cv      = skf,
    scoring = "accuracy",
    n_jobs  = -1
)

print(f"CV Scores : {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean      : {cv_scores.mean()*100:.2f}%")
print(f"Std Dev   : {cv_scores.std()*100:.2f}%  {'- Stable' if cv_scores.std() < 0.02 else '- High variance'}")

CV Scores : ['0.8309', '0.8219', '0.8230', '0.8177', '0.8351']
Mean      : 82.57%
Std Dev   : 0.63%  - Stable
